# YARN and MapReduce on OpenShift

The shell cells are complete VM-terminal command blocks. Copy each block into the VM terminal; the in-cluster Jupyter container does not provide the `oc` administration client. Commands use the current OpenShift context and namespace `bigdata`.


## Initialize the VM terminal

Open a terminal in the Linux VM. The one-time functions are installed by `00-Introduction/02-Installation.ipynb`. Load them, connect `oc` to the running CRC cluster, and select the `bigdata` project before continuing.

In [ ]:
source "$HOME/.bashrc"
if ! declare -F okd-env >/dev/null; then
  echo 'Complete 00-Introduction/02-Installation.ipynb first.' >&2
  return 1 2>/dev/null || exit 1
fi
okd-env
oc whoami
oc project
oc get nodes


## 1. Start HDFS and the optional YARN services

The default low-memory profile keeps YARN stopped. This lesson starts one ResourceManager, one NodeManager and one MapReduce History Server.


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata scale statefulset/namenode statefulset/datanode --replicas=1
oc -n bigdata rollout status statefulset/namenode --timeout=300s
oc -n bigdata rollout status statefulset/datanode --timeout=300s
oc -n bigdata scale deployment/resourcemanager deployment/nodemanager deployment/historyserver --replicas=1
for d in resourcemanager nodemanager historyserver; do oc -n bigdata rollout status deployment/$d --timeout=300s; done
oc -n bigdata get pods -l 'app in (resourcemanager,nodemanager,historyserver)' -o wide


## 2. Inspect YARN Services and registered nodes


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata get deployment,service -l 'app in (resourcemanager,nodemanager,historyserver)' -o wide
oc -n bigdata exec deployment/resourcemanager -- yarn node -list -all
oc -n bigdata exec deployment/resourcemanager -- yarn application -list -appStates ALL


## 3. Create word-count input in HDFS


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata exec namenode-0 -- sh -c "printf 'hadoop spark hive\nhadoop yarn spark\nopenshift hadoop\n' > /tmp/words.txt"
oc -n bigdata exec namenode-0 -- hdfs dfs -mkdir -p /training/mapreduce/input
oc -n bigdata exec namenode-0 -- hdfs dfs -put -f /tmp/words.txt /training/mapreduce/input/words.txt
oc -n bigdata exec namenode-0 -- hdfs dfs -rm -r -f /training/mapreduce/output
oc -n bigdata exec namenode-0 -- hdfs dfs -cat /training/mapreduce/input/words.txt


## 4. Run MapReduce word count


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata exec deployment/resourcemanager -- hadoop jar /opt/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.3.6.jar wordcount /training/mapreduce/input /training/mapreduce/output
oc -n bigdata exec namenode-0 -- hdfs dfs -ls /training/mapreduce/output
oc -n bigdata exec namenode-0 -- hdfs dfs -cat /training/mapreduce/output/part-r-00000


## 5. Run the MapReduce Pi example


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata exec deployment/resourcemanager -- hadoop jar /opt/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.3.6.jar pi 2 100


## 6. Inspect applications and logs


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata exec deployment/resourcemanager -- yarn application -list -appStates ALL
APP_ID=$(oc -n bigdata exec deployment/resourcemanager -- yarn application -list -appStates ALL 2>/dev/null | awk '/application_/ {print $1}' | tail -1)
echo "Latest application: ${APP_ID:-none}"
if [[ -n "${APP_ID:-}" ]]; then
  oc -n bigdata exec deployment/resourcemanager -- yarn application -status "$APP_ID" || true
fi


## 7. Open YARN web interfaces locally


In [ ]:
%%bash
set -euo pipefail
echo 'ResourceManager: oc -n bigdata port-forward --address=127.0.0.1 service/resourcemanager 18088:8088'
echo 'Open: http://127.0.0.1:18088'
echo
echo 'NodeManager: oc -n bigdata port-forward --address=127.0.0.1 service/nodemanager 18042:8042'
echo 'Open: http://127.0.0.1:18042'
echo
echo 'History Server: oc -n bigdata port-forward --address=127.0.0.1 service/historyserver 19888:19888'
echo 'Open: http://127.0.0.1:19888'


## 8. Restart or stop YARN

HDFS data remains available when YARN is stopped.


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata rollout restart deployment/resourcemanager deployment/nodemanager deployment/historyserver
for d in resourcemanager nodemanager historyserver; do oc -n bigdata rollout status deployment/$d --timeout=300s; done
oc -n bigdata exec deployment/resourcemanager -- yarn node -list -all


In [ ]:
%%bash
set -euo pipefail
echo 'Stop YARN after this lesson:'
oc -n bigdata scale deployment/resourcemanager deployment/nodemanager deployment/historyserver --replicas=0
oc -n bigdata get deployment resourcemanager nodemanager historyserver
